# Proyecto de aplicación de todo lo previo

In [1]:
#Generación del dataframe inicial
import pandas as pd
import numpy as np

np.random.seed(42)
n_ventas = 1500
n_usuarios = 800

# Dataset 1: Ventas brutas
df_ventas_raw = pd.DataFrame({
    'id_transaccion': range(5001, 5001 + n_ventas),
    'id_cliente': np.random.randint(1001, 1001 + n_usuarios + 50, n_ventas), # Algunos clientes no existirán en la base de usuarios
    'producto': np.random.choice(['Laptop', 'Smartphone', 'Teclado', 'Monitor'], n_ventas),
    'precio': np.random.choice([1200, 800, 25, 300, 15000], n_ventas, p=[0.4, 0.4, 0.1, 0.09, 0.01]), # Incluye un outlier de 15000
    'fecha': pd.date_range(start='2026-01-01', periods=n_ventas, freq='min'),
    'tienda': np.random.choice([' Madrid', 'madrid', 'BARCELONA', 'Barcelona ', 'Valencia'], n_ventas)
})

# Dataset 2: Información de usuarios
df_clientes_raw = pd.DataFrame({
    'id_cliente': range(1001, 1001 + n_usuarios),
    'edad': np.random.choice([20, 34, 45, np.nan, 29, 60, np.nan], n_usuarios),
    'nivel_suscripcion': np.random.choice(['Premium', 'Basic', 'Free'], n_usuarios)
})

# Introducir duplicados artificiales en ventas
df_ventas_raw = pd.concat([df_ventas_raw, df_ventas_raw.iloc[:30]], ignore_index=True)
print("Datasets listos para el proyecto.")

Datasets listos para el proyecto.


## Fase 1: Consolidación y Diagnóstico inicial

### Unión de Tablas

In [2]:
df_consolidada = pd.merge(df_ventas_raw, df_clientes_raw, how = 'left', on = 'id_cliente')
print(df_consolidada.head())

   id_transaccion  id_cliente    producto  precio               fecha  \
0            5001        1103     Teclado    1200 2026-01-01 00:00:00   
1            5002        1436  Smartphone     300 2026-01-01 00:01:00   
2            5003        1271     Teclado      25 2026-01-01 00:02:00   
3            5004        1107     Teclado    1200 2026-01-01 00:03:00   
4            5005        1072     Teclado     800 2026-01-01 00:04:00   

       tienda  edad nivel_suscripcion  
0   BARCELONA  20.0              Free  
1      madrid   NaN           Premium  
2      Madrid  45.0             Basic  
3    Valencia  34.0           Premium  
4  Barcelona    NaN             Basic  


# Fase 2: Limpieza de datos 

In [5]:
print(f"Las filas totales del DataFrame son {len(df_consolidada)}")
df_consolidada.isnull().sum() #Vamos los datos que son NaN en cada variable

Las filas totales del DataFrame son 1530


id_transaccion         0
id_cliente             0
producto               0
precio                 0
fecha                  0
tienda                 0
edad                 485
nivel_suscripcion    103
dtype: int64

In [8]:
df_consolidada.drop_duplicates(inplace = True)
print(f"Número de registros del DF sin duplicados: {len(df_consolidada)}.")

Número de registros del DF sin duplicados: 1500.


In [9]:
mediana_edad = df_consolidada['edad'].median()
df_consolidada.fillna({'edad' : mediana_edad, 'nivel_suscripcion' : 'Desconocido'}, inplace = True)
df_consolidada.isnull().sum()

id_transaccion       0
id_cliente           0
producto             0
precio               0
fecha                0
tienda               0
edad                 0
nivel_suscripcion    0
dtype: int64

In [ ]:
df_consolidada['tienda'].unique()

array(['BARCELONA', 'madrid', ' Madrid', 'Valencia', 'Barcelona '],
      dtype=object)

In [ ]:
df_consolidada['tienda'] = df_consolidada['tienda'].str.strip().str.title() 
#Elimina caracteres especiales antes y después de una palabra, después pone todas en minúscula menos la primera letra.
print(df_consolidada['tienda'].unique())

['Barcelona' 'Madrid' 'Valencia']


In [16]:
#Eliminación de filas con valores atípicos (outliers)
q1 = df_consolidada['precio'].quantile(0.25)
q3 = df_consolidada['precio'].quantile(0.75)
iqr = q3-q1
lim_inf = q1 - iqr * 1.5
lim_sup = q3 + iqr * 1.5
df_final = df_consolidada[(df_consolidada['precio'] >= lim_inf) & (df_consolidada['precio'] <= lim_sup)].copy()


In [17]:
print(f"Finalmente quedan {len(df_final)} registros en el DataFrame final.")

Finalmente quedan 1317 registros en el DataFrame final.


## Fase 3: Creación de variables.

In [18]:
#Segmentación de Variables
df_final['rango_edad'] = pd.cut(df_final['edad'], bins= [0, 30, 50, 100], labels = ['Joven', 'Adulto', 'Senior'])
print(df_final[['edad', 'rango_edad']].head())

   edad rango_edad
0  20.0      Joven
1  34.0     Adulto
3  34.0     Adulto
4  34.0     Adulto
5  34.0     Adulto
